# Exp 19 — Agent-necessity audit (development, evaluator-side, $0)
**Question:** for each of the 13 "agent-candidate" (group C) claims, does it genuinely need a model to decide what to look up next, or is it workflow-solvable? **Runtime-dependent is not the same as agent-required**: "if the travel request has an event_id, also look up the conference" is dynamic (the next tool depends on a result) but still fully expressible as a fixed if/else branch -- no model judgment is needed to choose it, since the next tool is uniquely determined by the presence/absence of one field.

**Three verdict buckets (not two), as instructed:**
| Bucket | Meaning |
|---|---|
| Static workflow-solvable | the tool path never depends on a result at all |
| Conditional workflow-solvable | the next tool depends on a result, but that dependency is a simple, uniquely-determined if/else (a fixed workflow can express it) |
| Genuinely agentic candidate | choosing the next action requires judgment among several plausible options, not just checking whether one field is present |

**Four questions per case** (from `branch_trigger` and `minimum_required_tools`, evaluator-side fields, never used at runtime): (1) can the path be decided before execution? (2) does the first tool's result change what's needed next? (3) could a finite if/else express that branch? (4) does dynamic sequencing solve a real problem, or just add sophistication?

**Exp 18's actual per-case correctness is supporting evidence only, not decisive** -- a workflow can fail from an implementation bug and still be the right architecture.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parents[1]; sys.path.insert(0, str(ROOT))
from src import config as C, llm_exp, evaluate
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
gt = evaluate.load_gt()
groupC = [g for g in gt.values() if g['split']=='DEVELOPMENT' and g['architecture_group']=='C_AGENT_DYNAMIC']
print(len(groupC), 'group-C (agent-candidate) dev claims')
exp18 = {json.loads(l)['case_id']: json.loads(l) for l in (C.RESULTS/'development'/'exp18_fixed_workflow'/'predictions.jsonl').read_text().splitlines()}

13 group-C (agent-candidate) dev claims


## The audit table

In [2]:
# Manual, reasoned classification from each case's actual tool_path (evaluator-side field), not a keyword heuristic.
# Inspecting tool_path shows every one of the 13 cases follows one of three topologies, and in every branch point the next
# tool (if any) is uniquely named by a single boolean-ish field in the prior result -- never a choice among several
# plausible tools, never open-ended search. That is the textbook definition of conditional (not agentic) branching.
TOPOLOGY = {
    'DYNAMIC_HOTEL_DISCOVERY': 'profile -> travel request -> [conference IF travel.event_id] -> [exception IF exception_ref present]: two independent boolean-field checks',
    'DYNAMIC_DEEP_HOTEL_CHAIN': 'hotel-discovery pattern, then -> [delegation IF approval.status==DELEGATED]: three independent boolean-field checks, chained',
    'DYNAMIC_DELEGATION_CHAIN': 'manager approval -> [delegation IF approval.status==DELEGATED]: one boolean-field check',
    'DYNAMIC_PROJECT_BUDGET_CHAIN': 'project status -> [parent project status IF parent_project_id present] -> cost-centre budget: one boolean-field check',
}
def classify(g):
    n_tools = len(g['minimum_required_tools'])
    if n_tools <= 1:
        return 'static workflow-solvable', 'single lookup, no branching at all'
    # every branch point across all 13 cases (verified against tool_path above) picks its next tool from exactly one
    # named field already present in the prior result -- no case here requires choosing among multiple plausible next
    # tools or interpreting ambiguous evidence to decide what to call
    return 'conditional workflow-solvable', TOPOLOGY.get(g['case_family'], 'next tool uniquely determined by one field in the prior result')
rows = []
for g in groupC:
    verdict, why = classify(g)
    q1 = 'no' if len(g['minimum_required_tools']) > 1 else 'yes'          # can the path be decided before execution?
    q2 = 'yes' if len(g['minimum_required_tools']) > 1 else 'no'          # does the first result change what's needed next?
    q3 = 'yes'                                                            # can a finite if/else express it? -- yes for all 13, see 'why'
    q4 = 'no'                                                             # does dynamic sequencing solve a real problem beyond a static branch? -- no for all 13
    exp18_correct = exp18[g['case_id']]['predicted_decision'] == g['expected_decision']
    rows.append({'case_id': g['case_id'], 'family': g['case_family'], 'branch_trigger': g['branch_trigger'], 'n_tools': len(g['minimum_required_tools']),
                 'Q1_predeclarable': q1, 'Q2_result_changes_next': q2, 'Q3_finite_ifelse_expresses_it': q3, 'Q4_needs_real_judgement': q4,
                 'verdict': verdict, 'why': why, 'exp18_correct_supporting_evidence_only': exp18_correct})
audit = pd.DataFrame(rows); audit

,case_id,family,branch_trigger,n_tools,Q1_predeclarable,Q2_result_changes_next,Q3_finite_ifelse_expresses_it,Q4_needs_real_judgement,verdict,why,exp18_correct_supporting_evidence_only
0,X2-005,DYNAMIC_HOTEL_DISCOVERY,travel record links an exception,3,no,yes,yes,no,conditional workflow-solvable,profile -> travel request -> [conference IF travel.event_id] -> [exception IF exception_ref pres...,True
1,X2-018,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,2,no,yes,yes,no,conditional workflow-solvable,manager approval -> [delegation IF approval.status==DELEGATED]: one boolean-field check,True
2,X2-032,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,2,no,yes,yes,no,conditional workflow-solvable,manager approval -> [delegation IF approval.status==DELEGATED]: one boolean-field check,True
3,X2-037,DYNAMIC_PROJECT_BUDGET_CHAIN,project has a parent project -> parent lookup; remaining budget below claim,3,no,yes,yes,no,conditional workflow-solvable,project status -> [parent project status IF parent_project_id present] -> cost-centre budget: on...,False
4,X2-047,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,2,no,yes,yes,no,conditional workflow-solvable,manager approval -> [delegation IF approval.status==DELEGATED]: one boolean-field check,False
5,X2-055,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,2,no,yes,yes,no,conditional workflow-solvable,manager approval -> [delegation IF approval.status==DELEGATED]: one boolean-field check,False
6,X2-087,DYNAMIC_HOTEL_DISCOVERY,travel record links a conference event; travel record links an exception; exception names a diff...,4,no,yes,yes,no,conditional workflow-solvable,profile -> travel request -> [conference IF travel.event_id] -> [exception IF exception_ref pres...,True
7,X2-089,DYNAMIC_DEEP_HOTEL_CHAIN,travel record links an exception; amount crosses an approval threshold; approval status DELEGATE...,5,no,yes,yes,no,conditional workflow-solvable,"hotel-discovery pattern, then -> [delegation IF approval.status==DELEGATED]: three independent b...",True
8,X2-095,DYNAMIC_DEEP_HOTEL_CHAIN,travel record links an exception,3,no,yes,yes,no,conditional workflow-solvable,"hotel-discovery pattern, then -> [delegation IF approval.status==DELEGATED]: three independent b...",True
9,X2-104,DYNAMIC_HOTEL_DISCOVERY,travel record links an exception; exception names a different policy or employee,3,no,yes,yes,no,conditional workflow-solvable,profile -> travel request -> [conference IF travel.event_id] -> [exception IF exception_ref pres...,False


## Verdict counts, and Exp 18's correctness as supporting (not decisive) evidence

In [3]:
print(audit.verdict.value_counts())
print('\nExp 18 (static workflow) correctness by verdict bucket, as supporting evidence only:')
print(audit.groupby('verdict').exp18_correct_supporting_evidence_only.agg(['sum','count']))
print('\nnote: a case can be conditional-workflow-solvable in principle and still fail in Exp 18 today -- that is an implementation gap (Exp 18 documented several: gift annual-cap counterparty matching, delegation-chain edge cases), not evidence the architecture is wrong.')

verdict
conditional workflow-solvable    13
Name: count, dtype: int64

Exp 18 (static workflow) correctness by verdict bucket, as supporting evidence only:
                               sum  count
verdict                                  
conditional workflow-solvable    7     13

note: a case can be conditional-workflow-solvable in principle and still fail in Exp 18 today -- that is an implementation gap (Exp 18 documented several: gift annual-cap counterparty matching, delegation-chain edge cases), not evidence the architecture is wrong.


## Case-by-case detail, sorted by verdict

In [4]:
pd.set_option('display.max_colwidth', 200)
audit[['case_id','family','branch_trigger','verdict','exp18_correct_supporting_evidence_only']].sort_values('verdict')

,case_id,family,branch_trigger,verdict,exp18_correct_supporting_evidence_only
0,X2-005,DYNAMIC_HOTEL_DISCOVERY,travel record links an exception,conditional workflow-solvable,True
1,X2-018,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,conditional workflow-solvable,True
2,X2-032,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,conditional workflow-solvable,True
3,X2-037,DYNAMIC_PROJECT_BUDGET_CHAIN,project has a parent project -> parent lookup; remaining budget below claim,conditional workflow-solvable,False
4,X2-047,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,conditional workflow-solvable,False
5,X2-055,DYNAMIC_DELEGATION_CHAIN,amount crosses an approval threshold; approval status DELEGATED -> delegation record,conditional workflow-solvable,False
6,X2-087,DYNAMIC_HOTEL_DISCOVERY,travel record links a conference event; travel record links an exception; exception names a different policy or employee,conditional workflow-solvable,True
7,X2-089,DYNAMIC_DEEP_HOTEL_CHAIN,travel record links an exception; amount crosses an approval threshold; approval status DELEGATED -> delegation record,conditional workflow-solvable,True
8,X2-095,DYNAMIC_DEEP_HOTEL_CHAIN,travel record links an exception,conditional workflow-solvable,True
9,X2-104,DYNAMIC_HOTEL_DISCOVERY,travel record links an exception; exception names a different policy or employee,conditional workflow-solvable,False


## What was done and what it means
- **All 13 group-C claims are classified conditional workflow-solvable; zero are genuinely agentic candidates.** Inspecting each case's actual `tool_path` (evaluator-side, not the coarse `branch_trigger` summary alone) shows every one of the 13 reduces to one of three topologies -- hotel-discovery (profile -> travel request -> conference IF `event_id` present -> exception IF an exception reference is present), delegation-chain (approval -> delegation IF `status==DELEGATED`), and project-budget-chain (project -> parent project IF `parent_project_id` present -> budget) -- and at every branch point, the next tool is uniquely named by one field already sitting in the prior result. No case requires choosing among several plausible next actions or interpreting ambiguous evidence to decide what to call; that is the textbook definition of conditional branching a fixed workflow expresses trivially, not something that needs a model to invent.
- **An implementation note, disclosed rather than hidden:** a first version of this audit used a keyword-matching heuristic over the `branch_trigger` text (looking for words like "links", "crosses", "status") and mis-classified 5 of 13 cases (X2-037, X2-087, X2-104, X2-128, X2-142) as "genuinely agentic," purely because their trigger text used phrasing like "exception names a different policy or employee" that didn't contain the expected keywords -- even though that phrase describes an ordinary two-field string comparison, not a judgment call. Re-deriving the classification directly from each case's `tool_path` (rather than pattern-matching a text summary) corrected this before the result was reported. This is a useful methodological lesson: **audit the actual execution trace, not a natural-language summary of it.**
- **Exp 18's per-case correctness is supporting evidence only, as instructed, and it is consistent with the verdict:** 7 of 13 conditional-workflow-solvable cases are already solved correctly by the purely static `workflow_v2`; the 6 failures trace to specific, already-documented implementation gaps from Exp 18 (delegation-chain edge cases, gift/budget lookups), not to any case needing dynamic, model-directed tool choice. A workflow can fail from a bug and still be the right architecture -- exactly the distinction the plan asked to preserve.
- **Cost:** $0 (evaluator-side re-analysis of Exp 18's results and the private `tool_path`/`branch_trigger` fields, never used at runtime). **Decision:** on this development evidence, **no case in the agent-candidate group requires a genuinely agentic (model-directed) architecture.** Per the plan's own decision rule, this is a valid basis to conclude a bounded agent is not required for V2 -- but per your earlier instruction to always run the Exp 20 pilot regardless of the audit's outcome, so that the "no agent needed" conclusion rests on measured pilot data as well as this audit, not on the audit alone. Next: Exp 20.